# Assigment 1

This notebook holds the: 
- Fetching
- Cloning
- Extraction of methods
- Cleaning/ filtering of methods
- Dividing to training, testing and validation sets

In [9]:
!pip install javalang gitpython pandas

In [10]:
import os
import glob
import subprocess
import json
import random
import shutil
from pathlib import Path

import pandas as pd
import javalang
from javalang.tokenizer import tokenize


In [11]:

# Configuration

CLONE_DIR = "dataset/java_repos"
OUTPUT_DIR = "dataset/ngram_dataset"

CLASSES_PER_REPO = 20   # Java files to sample per repo
MIN_TOKENS = 10         # Minimum tokens per method
MAX_TOKENS = 512
MIN_UNIQUE_TOKENS = 10

VAL_SIZE = 1000
TEST_SIZE = 1000

# Create directories
os.makedirs(CLONE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete!")
print(f"Clone directory: {CLONE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Setup complete!
Clone directory: dataset/java_repos
Output directory: dataset/ngram_dataset


In [12]:
import requests

def fetch_top_java_repos(num_repos=200, per_page=100):
    """
    Fetch top-starred Java repositories from GitHub API.
    Skips forked repos to avoid duplicate code.
    Only get pushed past 2020 for modern up to date code
    """
    repos = []
    page = 1

    while len(repos) < num_repos:
        url = "https://api.github.com/search/repositories"
        params = {
          # Added stars > 500 for higher quality and pushed date for modern code
            "q": "language:java stars:>500 pushed:>2020-01-01",
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            break

        data = response.json()
        items = data.get("items", [])

        if not items:
            break

        for item in items:
            if item.get("fork", False):
                continue

            repos.append({
                "full_name": item["full_name"],
                "clone_url": item["clone_url"],
                "stars": item["stargazers_count"],
                "description": item.get("description", "")
            })

        page += 1

        if len(repos) >= num_repos:
            break

    return repos[:num_repos]

# Fetch repositories
print("Fetching top Java repositories from GitHub...")
repo_data = fetch_top_java_repos(num_repos=700)
df_repos = pd.DataFrame(repo_data)

print(f"\nFetched {len(df_repos)} repositories")
print(f"\nTop 10 repos by stars:")
df_repos.head(10)

Fetching top Java repositories from GitHub...

Fetched 700 repositories

Top 10 repos by stars:


,full_name,clone_url,stars,description
0,Snailclimb/JavaGuide,https://github.com/Snailclimb/JavaGuide.git,153809,Java 面试 & 后端通用面试指南，覆盖计算机基础、数据库、分布式、高并发与系统设计。准备...
1,krahets/hello-algo,https://github.com/krahets/hello-algo.git,122318,《Hello 算法》：动画图解、一键运行的数据结构与算法教程。支持简中、繁中、English...
2,GrowingGit/GitHub-Chinese-Top-Charts,https://github.com/GrowingGit/GitHub-Chinese-T...,106184,:cn: GitHub中文排行榜，各语言分设「软件 | 资料」榜单，精准定位中文好项目。各取...
3,iluwatar/java-design-patterns,https://github.com/iluwatar/java-design-patter...,93749,Design patterns implemented in Java
4,macrozheng/mall,https://github.com/macrozheng/mall.git,82919,mall项目是一套电商系统，包括前台商城系统及后台管理系统，基于Spring Boot+My...
5,spring-projects/spring-boot,https://github.com/spring-projects/spring-boot...,79935,Spring Boot helps you to create Spring-powered...
6,doocs/advanced-java,https://github.com/doocs/advanced-java.git,78845,😮 Core Interview Questions & Answers For Exper...
7,MisterBooo/LeetCodeAnimation,https://github.com/MisterBooo/LeetCodeAnimatio...,76711,Demonstrate all the questions on LeetCode in t...
8,elastic/elasticsearch,https://github.com/elastic/elasticsearch.git,76073,"Free and Open Source, Distributed, RESTful Sea..."
9,TheAlgorithms/Java,https://github.com/TheAlgorithms/Java.git,65057,All Algorithms implemented in Java


---
## Clone Repositories

We clone each repository using `--depth 1` (shallow clone) since we only need the current code snapshot, not the full commit history. This saves time and disk space.

In [13]:
def clone_repo(clone_url, dest_dir):
    """
    Shallow clone a repository.
    Returns True if successful, False otherwise.
    """
    try:
        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)

        cmd = ["git", "clone", "--depth", "1", "--quiet", clone_url, dest_dir]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)

        return result.returncode == 0
    except subprocess.TimeoutExpired:
        print(f"  Timeout cloning {clone_url}")
        return False
    except Exception as e:
        print(f"  Error: {e}")
        return False


# Clone repositories
cloned_repos = []
failed_repos = []

print(f"Cloning {len(df_repos)} repositories...\n")

for idx, row in df_repos.iterrows():
    repo_name = row["full_name"]
    clone_url = row["clone_url"]

    safe_name = repo_name.replace("/", "_")
    dest_dir = os.path.join(CLONE_DIR, safe_name)

    print(f"[{idx+1}/{len(df_repos)}] Cloning {repo_name}...", end=" ")

    success = clone_repo(clone_url, dest_dir)

    if success:
        cloned_repos.append({
            "repo_name": repo_name,
            "local_path": dest_dir,
            "stars": row["stars"]
        })
        print("done")
    else:
        failed_repos.append(repo_name)
        print("failed")

print(f"\n\nSummary:")
print(f"  Successfully cloned: {len(cloned_repos)}")
print(f"  Failed: {len(failed_repos)}")

Cloning 700 repositories...

[1/700] Cloning Snailclimb/JavaGuide... done
[2/700] Cloning krahets/hello-algo... done
[3/700] Cloning GrowingGit/GitHub-Chinese-Top-Charts... done
[4/700] Cloning iluwatar/java-design-patterns... done
[5/700] Cloning macrozheng/mall... done
[6/700] Cloning spring-projects/spring-boot... done
[7/700] Cloning doocs/advanced-java... done
[8/700] Cloning MisterBooo/LeetCodeAnimation... done
[9/700] Cloning elastic/elasticsearch... done
[10/700] Cloning TheAlgorithms/Java... done
[11/700] Cloning kdn251/interviews... done
[12/700] Cloning NationalSecurityAgency/ghidra... done
[13/700] Cloning spring-projects/spring-framework... done
[14/700] Cloning google/guava... done
[15/700] Cloning termux/termux-app... done
[16/700] Cloning dbeaver/dbeaver... done
[17/700] Cloning ReactiveX/RxJava... done
[18/700] Cloning skylot/jadx... done
[19/700] Cloning jeecgboot/JeecgBoot... done
[20/700] Cloning apache/dubbo... done
[21/700] Cloning PhilJay/MPAndroidChart... done
[

---
## Find and Select Java Files

For each cloned repository, we:
1. Find all `.java` files (excluding test/example directories)
2. Randomly select up to 20 files per repo
3. Track which files were used (so students can build test sets from remaining files)

In [14]:
def find_java_files(repo_path):
    """
    Find all .java files in a repository.
    Excludes test files and common non-source directories.
    """
    java_files = []
    exclude_patterns = ["test", "tests", "example", "examples", "sample", "demo", "generated"]

    for root, dirs, files in os.walk(repo_path):
        root_lower = root.lower()
        if any(pattern in root_lower for pattern in exclude_patterns):
            continue

        for file in files:
            if file.endswith(".java"):
                java_files.append(os.path.join(root, file))

    return java_files


def select_java_files(java_files, max_files):
    """
    Randomly select up to max_files from the list.
    """
    if len(java_files) <= max_files:
        return java_files
    return random.sample(java_files, max_files)


# Find and select Java files from each repo
repo_java_files = {}
all_selected_files = []

print(f"Finding Java files (selecting up to {CLASSES_PER_REPO} per repo)...\n")

for repo_info in cloned_repos:
    repo_name = repo_info["repo_name"]
    repo_path = repo_info["local_path"]

    java_files = find_java_files(repo_path)

    if not java_files:
        print(f"  {repo_name}: No Java files found")
        continue

    selected = select_java_files(java_files, max_files=CLASSES_PER_REPO)

    repo_java_files[repo_name] = {
        "total_files": len(java_files),
        "selected_files": [os.path.relpath(f, repo_path) for f in selected],
        "remaining_files": len(java_files) - len(selected)
    }

    all_selected_files.extend([(repo_name, f) for f in selected])
    print(f"  {repo_name}: {len(selected)}/{len(java_files)} files selected")

print(f"\nTotal Java files selected: {len(all_selected_files)}")

Finding Java files (selecting up to 20 per repo)...

  Snailclimb/JavaGuide: 1/1 files selected
  krahets/hello-algo: 20/340 files selected
  GrowingGit/GitHub-Chinese-Top-Charts: 1/1 files selected
  iluwatar/java-design-patterns: 20/1262 files selected
  macrozheng/mall: 20/508 files selected
  spring-projects/spring-boot: 20/3725 files selected
  doocs/advanced-java: 1/1 files selected
  MisterBooo/LeetCodeAnimation: 11/11 files selected
  elastic/elasticsearch: 20/14108 files selected
  TheAlgorithms/Java: 20/786 files selected
  kdn251/interviews: 20/516 files selected
  NationalSecurityAgency/ghidra: 20/13267 files selected
  spring-projects/spring-framework: 20/4929 files selected
  google/guava: 20/1270 files selected
  termux/termux-app: 20/174 files selected
  dbeaver/dbeaver: 20/5985 files selected
  ReactiveX/RxJava: 20/914 files selected
  skylot/jadx: 20/1209 files selected
  jeecgboot/JeecgBoot: 20/786 files selected
  apache/dubbo: 20/2390 files selected
  PhilJay/MPAnd

---
## Parse and Extract Methods

For each Java file, we:
1. Parse the source code using `javalang`
2. Extract all method declarations
3. Store the method body along with metadata (repo, file, method name)

In [15]:
def read_file_content(file_path):
    """Read file content with multiple encoding fallbacks."""
    encodings = ['utf-8', 'latin-1', 'cp1252']

    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue

    return None


def extract_method_source(source_code, method_node, lines):
    """Extract the source code of a method by counting braces."""
    try:
        start_line = method_node.position.line - 1

        brace_count = 0
        started = False
        end_line = start_line

        for i in range(start_line, len(lines)):
            line = lines[i]
            for char in line:
                if char == '{':
                    brace_count += 1
                    started = True
                elif char == '}':
                    brace_count -= 1

            if started and brace_count == 0:
                end_line = i
                break

        method_lines = lines[start_line:end_line + 1]
        return '\n'.join(method_lines)

    except Exception:
        return None


def extract_methods_from_file(file_path, repo_name):
    """Parse a Java file and extract all methods."""
    methods = []

    source_code = read_file_content(file_path)
    if source_code is None:
        return methods

    lines = source_code.split('\n')

    try:
        tree = javalang.parse.parse(source_code)

        for path, node in tree.filter(javalang.tree.MethodDeclaration):
            method_source = extract_method_source(source_code, node, lines)

            if method_source:
                methods.append({
                    "repo": repo_name,
                    "file": os.path.basename(file_path),
                    "method_name": node.name,
                    "source": method_source
                })

    except javalang.parser.JavaSyntaxError:
        pass
    except Exception:
        pass

    return methods


# Extract methods from all selected files
all_methods = []

print(f"Extracting methods from {len(all_selected_files)} files...\n")

for i, (repo_name, file_path) in enumerate(all_selected_files):
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(all_selected_files)} files...")

    methods = extract_methods_from_file(file_path, repo_name)
    all_methods.extend(methods)

print(f"\nTotal methods extracted: {len(all_methods)}")

Extracting methods from 12274 files...

  Processed 100/12274 files...
  Processed 200/12274 files...
  Processed 300/12274 files...
  Processed 400/12274 files...
  Processed 500/12274 files...
  Processed 600/12274 files...
  Processed 700/12274 files...
  Processed 800/12274 files...
  Processed 900/12274 files...
  Processed 1000/12274 files...
  Processed 1100/12274 files...
  Processed 1200/12274 files...
  Processed 1300/12274 files...
  Processed 1400/12274 files...
  Processed 1500/12274 files...
  Processed 1600/12274 files...
  Processed 1700/12274 files...
  Processed 1800/12274 files...
  Processed 1900/12274 files...
  Processed 2000/12274 files...
  Processed 2100/12274 files...
  Processed 2200/12274 files...
  Processed 2300/12274 files...
  Processed 2400/12274 files...
  Processed 2500/12274 files...
  Processed 2600/12274 files...
  Processed 2700/12274 files...
  Processed 2800/12274 files...
  Processed 2900/12274 files...
  Processed 3000/12274 files...
  Process

---
## Filter Methods

We filter out methods that:
1. Contain non-ASCII characters (ensures clean tokenization)
2. Have fewer than 10 tokens (too short to be meaningful)
3. Remove duplicates
4. Check for uniqueness in each method
5. Have more than a max token of 512 tokens per method


In [ ]:
from pygments import token
def contains_non_ascii(text):
    """Check if text contains non-ASCII characters."""
    try:
        text.encode('ascii')
        return False
    except UnicodeEncodeError:
        return True


def count_tokens(source_code):
    """Count the number of Java tokens in source code."""
    try:
        tokens = list(tokenize(source_code))
        return len(tokens)
    except:
        return 0


# TODO: Write your filtering functions here

def unique_tokens(source_code):
  """List all the Java tokens in the source code."""
  try:
      tokens = list(tokenize(source_code))
      unique = set()
      for t in tokens:
        if hasattr(t, 'value'):
          unique.add(t.value)
      return unique
  except:
      return set()

def max_tokens(method):
  '''drop token if the length is too long,
  if too long could be poorly writtten '''
  try:
      tokens = list(tokenize(method))
      if len(tokens) > 512:
          return True
      return False
  except:
      return False

# Apply filters
filtered_methods = []
# Initialize a set to track source code we have already processed
seen_source_codes = set()

# Add a stat for duplicates
stats = {
    "total": len(all_methods),
    "duplicates_dropped": 0,
    "non_ascii_dropped": 0,
    "too_short_dropped": 0,
    "not_unique_dropped": 0,
    "max_tokens_dropped": 0,
    "kept": 0
}

print(f"Filtering {len(all_methods)} methods...\n")

for method in all_methods:
    source = method["source"]

    # Duplicates
    if source in seen_source_codes:
        stats["duplicates_dropped"] += 1
        continue

    # Non-ascii
    if contains_non_ascii(source):
        stats["non_ascii_dropped"] += 1
        continue

    # Min length
    token_count = count_tokens(source)
    if token_count < MIN_TOKENS:
        stats["too_short_dropped"] += 1
        continue
    # Max length
    if max_tokens(source):
        stats["max_tokens_dropped"] += 1
        continue

    # Uniqueness
    token_set = unique_tokens(source)
    if len(token_set) < MIN_UNIQUE_TOKENS:
        stats["not_unique_dropped"] += 1
        continue

    # If it reaches here, it passed all filters.
    # Add it to the seen set so the next identical method is dropped
    seen_source_codes.add(source)

    method["token_count"] = token_count
    filtered_methods.append(method)
    stats["kept"] += 1

# Updated Results Printout
print(f"Filtering Results:")
print(f"  Total methods:         {stats['total']}")
print(f"  Dropped (Duplicates):  {stats['duplicates_dropped']}")
print(f"  Dropped (non-ASCII):   {stats['non_ascii_dropped']}")
print(f"  Dropped (< {MIN_TOKENS} tokens): {stats['too_short_dropped']}")
print(f"  Dropped (< {MIN_UNIQUE_TOKENS} unique): {stats['not_unique_dropped']}")
print(f"  Dropped (> {MAX_TOKENS} tokens): {stats['max_tokens_dropped']}")
print(f"  -------------------------")
print(f"  Methods kept:          {stats['kept']}")

Filtering 99103 methods...

Filtering Results:
  Total methods:         99103
  Dropped (Duplicates):  5371
  Dropped (non-ASCII):   3175
  Dropped (< 10 tokens): 5485
  Dropped (< 10 unique): 702
  Dropped (> 512 tokens): 761
  -------------------------
  Methods kept:          83609


---
## Tokenize Methods

We tokenize each method into space-separated tokens using `javalang.tokenizer`.

Output format:
```
public void setName ( String name ) { this . name = name ; }
```

In [17]:
def tokenize_method(source_code):
    """Tokenize Java source code into space-separated tokens."""
    try:
        tokens = list(tokenize(source_code))
        token_values = [token.value for token in tokens]
        return ' '.join(token_values)
    except:
        return None


# Tokenize all methods
tokenized_methods = []

print(f"Tokenizing {len(filtered_methods)} methods...\n")

for method in filtered_methods:
    tokenized = tokenize_method(method["source"])

    if tokenized:
        tokenized_methods.append({
            "repo": method["repo"],
            "file": method["file"],
            "method_name": method["method_name"],
            "tokenized_code": tokenized,
            "token_count": method["token_count"]
        })

print(f"Successfully tokenized: {len(tokenized_methods)} methods")

# Show example
print(f"\nExample tokenized method:")
if tokenized_methods:
    example = tokenized_methods[0]
    print(f"  Repo: {example['repo']}")
    print(f"  File: {example['file']}")
    print(f"  Method: {example['method_name']}")
    print(f"  Tokens ({example['token_count']}):")
    print(f"  {example['tokenized_code'][:200]}..." if len(example['tokenized_code']) > 200 else f"  {example['tokenized_code']}")

Tokenizing 83609 methods...

Successfully tokenized: 83609 methods

Example tokenized method:
  Repo: Snailclimb/JavaGuide
  File: TranslateRepo.java
  Method: printHeader
  Tokens (53):
  private static void printHeader ( ) { System . out . println ( "=" . repeat ( 70 ) ) ; System . out . println ( "Repository Documentation Translation Tool" ) ; System . out . println ( "=" . repeat ( ...


---
## Clean, Deduplicate, and Split

We clean the dataset by removing:
- Methods with multiple method signatures in one line
- Incomplete methods (not ending with `}`)

Then we remove exact duplicates and split into train/val/test sets.

In [ ]:
import random

def is_clean_method(tokenized_code):
    """Check if method is clean (single method, complete)."""
    method_keywords = tokenized_code.count("public ") + tokenized_code.count("private ") + tokenized_code.count("protected ")
    if method_keywords > 1:
        return False
    if not tokenized_code.endswith("}"):
        return False
    return True

# Clean
print(f"Before cleaning: {len(tokenized_methods)}")
tokenized_methods = [m for m in tokenized_methods if is_clean_method(m['tokenized_code'])]
print(f"After cleaning: {len(tokenized_methods)}")

# Deduplicate
seen = set()
unique_methods = []
for m in tokenized_methods:
    if m['tokenized_code'] not in seen:
        seen.add(m['tokenized_code'])
        unique_methods.append(m)

print(f"After dedup: {len(unique_methods)}")
tokenized_methods = unique_methods

# Extract the tokenized strings for the model
# We keep them as strings here; split() will be used during N-gram counting
all_data = [m['tokenized_code'] for m in tokenized_methods]

# Shuffle to mix repositories for clean splitting
random.seed(42) 
random.shuffle(all_data)

# Slice Test and Validation
# Take these from the beginning to ensure they never enter the training set
test_set = all_data[:1000]
val_set = all_data[1000:2000]

# Create the Training sets 
# T1 is the first 15k, T2 is the first 25k T3 is the first 35k
train_pool = all_data[2000:]

T1 = train_pool[:15000]
T2 = train_pool[:25000]
T3 = train_pool[:35000]

print(f"Splitting Complete:")
print(f"  Test Set (Te): {len(test_set)}")
print(f"  Validation (V): {len(val_set)}")
print(f"  Training T1:    {len(T1)}")
print(f"  Training T2:    {len(T2)}")
print(f"  Training T3:    {len(T3)}")

Before cleaning: 77713
After cleaning: 77713
After dedup: 77713
Splitting Complete:
  Test Set (Te): 1000
  Validation (V): 1000
  Training T1:    15000
  Training T2:    25000
  Training T3:    35000


In [ ]:
'''Save the data files in seperate .txt so that they can be loaded in '''
def save_split(data, filename):
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filename, 'w', encoding='utf-8') as f:
        for method in data:
            f.write(method + '\n')
    return filepath

print("Saving dataset files ... \n")
t1_filepath = save_split(T1, 'train_15k.txt')
print(f" Saved {t1_filepath}")

t2_filepath = save_split(T2, 'train_25k.txt')
print(f" Saved {t2_filepath}")

t3_filepath = save_split(T3, 'train_35k.txt')
print(f" Saved {t3_filepath}")

val_filepath = save_split(val_set, 'validation.txt')
print(f" Saved {val_filepath}")

test_filepath = save_split(test_set, 'test_self_mined.txt')
print(f" Saved {test_filepath}")

Saving dataset files ... 

 Saved dataset/ngram_dataset/train_15k.txt
 Saved dataset/ngram_dataset/train_25k.txt
 Saved dataset/ngram_dataset/train_35k.txt
 Saved dataset/ngram_dataset/validation.txt
 Saved dataset/ngram_dataset/test_self_mined.txt
